# Building a Production-Grade AI Solution

## From Scripts to Systems: The Automated Corporate Brochure Generator

### THE ARCHITECTURAL CHALLENGE:

**Goal**: Given a company's URL, automatically crawl, filter, and synthesize a professional corporate brochure suitable for investors and stakeholders.

**Key Engineering Patterns**:
1. **Agentic Link Filtering**: Using an LLM to make decisions on which data to ingest.
2. **Structured Output Parsing**: Ensuring our LLM returns valid JSON for programmatic use.
3. **Content Aggregation**: Managing context windows while merging data from multiple sources.
4. **Streaming UI**: Implementing real-time feedback for long-running synthesis tasks.

In [32]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Our custom utility module for web scraping
from web_scraper import scrape_text_content, scrape_hyperlinks

client = OpenAI()
LLM_MODEL = "gpt-4o-mini"

def validate_environment_setup() -> None:
    """
    Verifies that all required API keys are properly configured in the environment.
    """

    # Explicitly load the .env file and check if it was successful
    env_loaded = load_dotenv(override=True)

    key_configs = {
        "OPENAI_API_KEY": "sk-",
        "ANTHROPIC_API_KEY": "sk-ant-",
        "GEMINI_API_KEY": "AIza",
        "DEEPSEEK_API_KEY": "sk-",
        "XAI_API_KEY": "xai-"
    }

    print("--- Environment Configuration Check ---")
    for key, expected_prefix in key_configs.items():
        value = os.getenv(key)

        if not value:
            print(f"❌ {key:<20}: Not found in .env")
            continue
        
        if not value.startswith(expected_prefix):
            print(f"{key:<20}: Found, but expected prefix '{expected_prefix}'")
        elif value.strip() != value:
            print(f"{key:<20}: Found, but contains hidden whitespace")
        else:
            print(f"{key:<20}: Validated (ends with ...{value[-4:]})")

validate_environment_setup()

--- Environment Configuration Check ---
OPENAI_API_KEY      : Validated (ends with ...HZwA)
ANTHROPIC_API_KEY   : Validated (ends with ...lwAA)
GEMINI_API_KEY      : Validated (ends with ...FkqI)
DEEPSEEK_API_KEY    : Validated (ends with ...50b7)
XAI_API_KEY         : Validated (ends with ...bQYY)


## Phase 1: Intelligent Link Analysis

Crawling every single link on a website is inefficient and consumes too many tokens. As **AI Architects**, we use a "Filter-First" approach. 

We use the LLM as a **Decision Engine** to identify high-value pages (About, Careers, Products) while ignoring noise (Terms of Service, Privacy Policy).

In [33]:
LINK_ANALYSIS_SYSTEM_PROMPT = """
You are a specialized Web Content Architect. Your task is to analyze a list of URLs from a company website 
and identify the most strategic pages for a corporate brochure.

Focus on:
- Company Overview / About Us
- Product/Service Catalogs
- Leadership/Team
- Career opportunities and culture

Output MUST be a valid JSON object following this schema:
{
    "strategic_links": [
        {"category": "About", "url": "https://example.com/about"},
        {"category": "Careers", "url": "https://example.com/jobs"}
    ]
}
"""

def construct_link_analysis_prompt(target_url):
    raw_links = scrape_hyperlinks(target_url)
    prompt = f"Target Website: {target_url}\n"
    prompt += "Extracted Raw Links (may be relative):\n"
    prompt += "\n".join(raw_links)
    prompt += "\n\nTask: Filter the above links. Ensure all returned URLs are absolute (start with https://)."

    return prompt
    
#construct_link_analysis_prompt("https://www.anthropic.com/")
#construct_link_analysis_prompt("https://www.google.com/")

In [34]:
def identify_strategic_links(url):
    print(f"🔍 Analyzing site structure for: {url}...")

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": LINK_ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": construct_link_analysis_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    analysis = json.loads(response.choices[0].message.content)
    links = analysis.get("strategic_links", [])
    print(f"🎯 Identified {len(links)} strategic pages.")
    print(f"📄 Strategic Links:")
    for link in links:
        print(f"  - {link['category']}: {link['url']}")

    return links

#identify_strategic_links("https://www.anthropic.com/")

## Phase 2: Knowledge Synthesis

Now that we have our high-value targets, we perform **Content Aggregation**. We fetch the text from each strategic page and present it to the LLM for final synthesis.


In [35]:
def compile_site_intelligence(base_url):
    # 1. Fetch Landing Page
    site_knowledge = f"# PRIMARY LANDING PAGE CONTENT:\n{scrape_text_content(base_url)}\n\n"

    # 2. Identify and Fetch Sub-pages
    strategic_pages = identify_strategic_links(base_url)

    for page in strategic_pages:
        category = page.get('category', 'Relevant Page')
        url = page.get('url')
        if url:
            print(f"📂 Ingesting {category}: {url}...")
            content = scrape_text_content(url)
            site_knowledge += f"\n# CONTENT FROM {category} PAGE ({url}):\n{content}\n"

    #print(site_knowledge)
    return site_knowledge

#compile_site_intelligence("https://www.anthropic.com/")

In [36]:
def construct_synthesis_prompt(company_name, site_knowledge):
    prompt = f"COMPANY: {company_name}\n\n"
    prompt += "Below is the raw intelligence gathered from the company website:\n"
    prompt += "---\n"
    prompt += site_knowledge[:10000] # Safe truncation for the mini model context window
    prompt += "\n---\n"
    prompt += "Construct the brochure based ONLY on the information above."
    return prompt

BROCHURE_SYNTHESIS_SYSTEM_PROMPT = """
You are an expert Corporate Communications Strategist.
Your goal is to synthesize raw website data into a compelling, professional corporate brochure.

Output Requirements:
- Use high-quality Markdown.
- Sections: Executive Summary, Core Offerings, Culture & Careers, and Future Outlook.
- Tone: Professional, authoritative, yet engaging.
- NO markdown code blocks (```) in the response.
"""

In [37]:
def render_corporate_brochure(company_name, url, stream=True):
    # Gather the intelligence
    raw_intelligence = compile_site_intelligence(url)

    # Prepare prompts
    user_input = construct_synthesis_prompt(company_name, raw_intelligence)
    
    print(f"🚀 Generating brochure for {company_name}...")

    if not stream:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": BROCHURE_SYNTHESIS_SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ]
        )
        display(Markdown(response.choices[0].message.content))
    else:
        stream_completion = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": BROCHURE_SYNTHESIS_SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ],
            stream=True
        )

    full_text = ""
    display_handle = display(Markdown("Synthesis in progress..."), display_id=True)

    for chunk in stream_completion:
        delta = chunk.choices[0].delta.content or ""
        full_text += delta
        update_display(Markdown(full_text), display_id=display_handle.display_id)


In [ ]:
# TEST THE PIPELINE
render_corporate_brochure("Anthropic", "https://www.anthropic.com")

🔍 Analyzing site structure for: https://www.anthropic.com...
🎯 Identified 4 strategic pages.
📄 Strategic Links:
  - About: https://www.anthropic.com/company
  - Products: https://claude.com/product/overview
  - Careers: https://www.anthropic.com/careers
  - Leadership: https://www.anthropic.com/leadership
📂 Ingesting About: https://www.anthropic.com/company...
📂 Ingesting Products: https://claude.com/product/overview...
📂 Ingesting Careers: https://www.anthropic.com/careers...
📂 Ingesting Leadership: https://www.anthropic.com/leadership...
🚀 Generating brochure for Anthropic...


# Anthropic Corporate Brochure

## Executive Summary
Anthropic is an innovative public benefit corporation dedicated to ensuring that artificial intelligence (AI) serves humanity’s long-term well-being. We are dedicated to building reliable, interpretable, and steerable AI systems that prioritize safety at the forefront of technological advancement. Our commitment goes beyond just research; we actively engage with the public, organizations, and governments worldwide to mitigate the risks associated with AI while maximizing its potential benefits.

## Core Offerings
### AI Models
Our flagship products, Claude and its variants (Claude Code, Claude Cowork, and Claude Platform), represent some of the most advanced AI capabilities available today. These tools empower businesses, nonprofits, and individuals through intuitive, reliable, and user-friendly platforms for coding, customer support, education, and various other applications.

- **Claude**: A powerful AI model designed to engage in genuinely helpful conversations.
- **Claude Code**: Tailored solutions for sophisticated coding assistance and security.
- **Claude Cowork**: Enhancing team collaboration across different tools like Slack, Excel, and PowerPoint.
- **Claude Platform**: A versatile environment catering to diverse user needs.

### Research and Development
At Anthropic, safety is treated as a science. Our interdisciplinary team conducts cutting-edge research across various modalities, exploring critical areas such as interpretability, reinforcement learning from human feedback, and the societal impacts of AI. Our results are consistently shared with the community to promote transparency and collaboration.

### Educational Initiatives
The Anthropic Academy provides learning resources, tutorials, and practical case studies designed to broaden understanding and effective utilization of AI technologies among diverse audiences.

## Culture & Careers
At Anthropic, our people are our greatest asset. We are a dynamic, collaborative team of researchers, engineers, policy experts, and operational leaders from various backgrounds, including NASA, military service, and startups. Our shared purpose is to shape the future responsibly.

Our core values reflect our commitment to excellence:
- **Act for the Global Good**: We prioritize actions that maximize positive outcomes for humanity.
- **Hold Light and Shade**: We recognize the balance between the risks and benefits of AI.
- **Be Good to Our Users**: Our customer-centric approach emphasizes kindness and support.
- **Ignite a Race to the Top on Safety**: We aspire to set industry standards on AI safety and encourage competitors to follow suit.

We welcome passionate individuals eager to contribute to a safer AI landscape.

## Future Outlook
The future of AI is fraught with both opportunities and challenges. Anthropic is poised to be at the forefront of this evolution, leading efforts in AI safety and reliability. As we continue to innovate and expand our offerings, we aim to collaborate with diverse stakeholders, including civil society organizations, government bodies, and academic institutions, to ensure that AI technology is developed and deployed responsibly.

By prioritizing safety and transparency, Anthropic aspires to not only be a part of the AI revolution but also to guide it towards a future that benefits all of humanity. Join us as we harness the potential of AI to make meaningful contributions to society.

Synthesis in progress...

UnboundLocalError: cannot access local variable 'stream_completion' where it is not associated with a value